# MLOps Training: Hands-On Session 1-5
## A Complete Journey from Data to Model Prediction

**Objective:** Understand MLOps by building a simple ML model end-to-end.

**What we'll cover:**
1. Load and explore data with Pandas
2. Clean and prepare data
3. Build a simple ML model
4. Track experiments with MLflow
5. Make predictions and evaluate

**Dataset:** Iris Flower Classification (beginner-friendly)
- 150 flowers with 4 features
- 3 classes (Setosa, Versicolor, Virginica)
- Goal: Predict flower type from measurements

## Step 1: Install Required Libraries

In [ ]:
# Install required packages
# Uncomment these if first time running

# !pip install pandas numpy scikit-learn matplotlib seaborn mlflow -q

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

print('✓ All libraries imported successfully!')

## Step 2: Load Data with Pandas

### What is Pandas?
- **Pandas** = Python Data Analysis Library
- Used to load, clean, and explore data
- Works with CSV, Excel, SQL, JSON, etc.
- DataFrame = like Excel spreadsheet in Python

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data  # Features (measurements)
y = iris.target  # Target (flower type)

# Create a Pandas DataFrame
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df['target_name'] = df['target'].map({
    0: 'Setosa',
    1: 'Versicolor',
    2: 'Virginica'
})

print('Dataset loaded successfully!')
print(f'Shape: {df.shape} (rows, columns)')
print(f'\nFirst 5 rows:')
print(df.head())

## Step 3: Explore and Understand Data

In [ ]:
# Check data types and missing values
print('Data Info:')
print(df.info())
print('\n' + '='*50)

# Statistical summary
print('\nStatistical Summary:')
print(df.describe())
print('\n' + '='*50)

# Check for missing values
print('\nMissing Values:')
print(df.isnull().sum())
print('\n✓ No missing values detected!')
print('\n' + '='*50)

# Check class distribution
print('\nClass Distribution:')
print(df['target_name'].value_counts())
print('✓ Balanced dataset (50 samples per class)')

## Step 4: Data Visualization

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Iris Dataset Exploration', fontsize=16, fontweight='bold')

# Plot 1: Distribution of Sepal Length
axes[0, 0].hist(df['sepal length (cm)'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Sepal Length Distribution')
axes[0, 0].set_xlabel('Sepal Length (cm)')
axes[0, 0].set_ylabel('Frequency')

# Plot 2: Distribution by Class
df.groupby('target_name')['sepal length (cm)'].mean().plot(kind='bar', ax=axes[0, 1], color=['red', 'green', 'blue'])
axes[0, 1].set_title('Average Sepal Length by Species')
axes[0, 1].set_ylabel('Sepal Length (cm)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Plot 3: Feature Correlation Heatmap
corr_matrix = df[df.columns[:-2]].corr()  # Exclude target columns
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 0], cbar=True)
axes[1, 0].set_title('Feature Correlation Matrix')

# Plot 4: Class Distribution
df['target_name'].value_counts().plot(kind='bar', ax=axes[1, 1], color=['red', 'green', 'blue'])
axes[1, 1].set_title('Class Distribution')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('data_exploration.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Visualization saved as data_exploration.png')

## Step 5: Prepare Data for Model

### Data Preparation Steps:
1. **Train/Test Split:** 70% training, 30% testing
2. **Feature Scaling:** Normalize features to [0,1] range
3. **Separate Features & Target:** X (features), y (labels)

In [ ]:
# Prepare data
X = df[df.columns[:-2]]  # Features (all columns except target columns)
y = df['target']  # Target labels

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print('\n' + '='*50)

# Split data: 70% train, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'\nTraining set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print('\n' + '='*50)

# Scale features (normalize to [0,1])
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\n✓ Data prepared successfully!')
print('  - Train/test split: 70/30')
print('  - Features scaled: StandardScaler')
print('  - Stratified split: balanced classes')

## Step 6: Build and Train ML Model

### What is Random Forest?
- **Ensemble method:** Combines 100 decision trees
- **How it works:** Each tree votes, majority wins
- **Why it's good:** Robust, handles non-linear relationships
- **Hyperparameters:** n_estimators (trees), max_depth (tree depth), random_state (reproducibility)

In [ ]:
# Initialize Random Forest model
model = RandomForestClassifier(
    n_estimators=100,  # Number of trees
    max_depth=5,       # Maximum tree depth
    random_state=42,   # For reproducibility
    n_jobs=-1          # Use all CPU cores
)

print('Model initialized:')
print(f'  - Algorithm: Random Forest Classifier')
print(f'  - n_estimators: {model.n_estimators}')
print(f'  - max_depth: {model.max_depth}')
print('\n' + '='*50)

# Train the model
print('\nTraining model...')
model.fit(X_train_scaled, y_train)
print('✓ Model training complete!')

## Step 7: Make Predictions and Evaluate

### Evaluation Metrics:
- **Accuracy:** (Correct predictions) / (Total predictions)
- **Precision:** Of predictions marked positive, how many were correct?
- **Recall:** Of actual positives, how many did we find?
- **F1 Score:** Harmonic mean of precision & recall (balanced metric)

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print('Model Performance Metrics:')
print('='*50)
print(f'Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')
print('='*50)

# Interpretation
if accuracy > 0.9:
    print('\n✓ Excellent model! (>90% accuracy)')
elif accuracy > 0.8:
    print('\n✓ Good model! (>80% accuracy)')
else:
    print('\n⚠ Model needs improvement')

## Step 8: Confusion Matrix - See Where Model Goes Wrong

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Setosa', 'Versicolor', 'Virginica'],
            yticklabels=['Setosa', 'Versicolor', 'Virginica'])
plt.title('Confusion Matrix - Model Predictions vs Actual', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print('Confusion Matrix Interpretation:')
print('  - Diagonal = Correct predictions')
print('  - Off-diagonal = Wrong predictions')
print('\nDiagonal values:')
for i, class_name in enumerate(['Setosa', 'Versicolor', 'Virginica']):
    print(f'  - {class_name}: {cm[i, i]} correct out of {cm[i].sum()} total')

## Step 9: Feature Importance - Which Features Matter Most?

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature Importance:')
print('='*50)
for idx, row in feature_importance.iterrows():
    print(f'{row["feature"]:25s}: {row["importance"]:.4f}')

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.xlabel('Importance Score', fontweight='bold')
plt.title('Feature Importance - Which Features Matter?', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nInterpretation:')
print(f'  - {feature_importance.iloc[0, 0]} is most important')
print(f'  - {feature_importance.iloc[-1, 0]} is least important')

## Step 10: Track with MLflow

### Why MLflow?
- **Reproducibility:** Every experiment is logged
- **Comparison:** Compare different models side-by-side
- **Model Registry:** Version control for models
- **Production Ready:** Track path to production

In [ ]:
# Set MLflow experiment
mlflow.set_experiment('Iris Classification')

# Start MLflow run
with mlflow.start_run():
    # Log hyperparameters
    mlflow.log_param('algorithm', 'Random Forest')
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('max_depth', 5)
    mlflow.log_param('test_size', 0.3)
    mlflow.log_param('random_state', 42)
    
    # Log metrics
    mlflow.log_metric('accuracy', accuracy)
    mlflow.log_metric('precision', precision)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('f1_score', f1)
    
    # Log model
    mlflow.sklearn.log_model(model, 'model')
    
    # Log artifacts (files)
    mlflow.log_artifact('data_exploration.png')
    mlflow.log_artifact('confusion_matrix.png')
    mlflow.log_artifact('feature_importance.png')
    
    # Log tags
    mlflow.set_tag('model_type', 'Classification')
    mlflow.set_tag('dataset', 'Iris')
    mlflow.set_tag('stage', 'Experiment')
    
    print('✓ MLflow tracking complete!')
    print('\nLogged to MLflow:')
    print('  - 5 hyperparameters')
    print('  - 4 metrics (accuracy, precision, recall, f1)')
    print('  - 1 trained model')
    print('  - 3 artifacts (plots)')
    print('  - 3 tags')
    print('\nTo view: mlflow ui → localhost:5000')

## Step 11: Make Single Predictions

### Real-world Usage: Get probability for a new flower

In [ ]:
# Example: Make prediction for a new flower
# Measurements: sepal_length=5.1, sepal_width=3.5, petal_length=1.4, petal_width=0.2

new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])
new_flower_scaled = scaler.transform(new_flower)

prediction = model.predict(new_flower_scaled)[0]
probabilities = model.predict_proba(new_flower_scaled)[0]

print('Prediction for New Flower:')
print('='*50)
print(f'Measurements: [5.1, 3.5, 1.4, 0.2]')
print(f'\nPredicted Class: {iris.target_names[prediction]}')
print(f'\nConfidence Scores:')
for idx, class_name in enumerate(iris.target_names):
    print(f'  - {class_name:15s}: {probabilities[idx]:.2%}')
print(f'\n✓ Model is {probabilities[prediction]:.2%} confident')

## Step 12: Summary & Key Takeaways

### What We Covered (5 Sessions in 1 Notebook):

**Session 1: MLOps Fundamentals**
- Understood that 95% of production ML is not the model

**Session 2: Technologies**
- Pandas (data handling), scikit-learn (ML algorithms)

**Session 3: Environment Setup**
- Libraries installed, data loaded, model trained

**Session 4: Data Pipelines**
- Explored data, cleaned it, split train/test, scaled features

**Session 5: Experiment Tracking**
- Tracked experiments with MLflow, logged metrics & models

In [ ]:
# Final Summary
print('='*60)
print('MLOPS TRAINING: COMPLETE SUMMARY')
print('='*60)
print(f'\n📊 Dataset: Iris (150 flowers, 3 classes)')
print(f'🔧 Model: Random Forest (100 trees)')
print(f'📈 Performance: {accuracy*100:.2f}% accuracy')
print(f'💾 Tracking: MLflow (5 params, 4 metrics, 1 model, 3 plots)')

print(f'\n📚 Key Learnings:')
print(f'  1. Pandas - Load and explore data')
print(f'  2. Scikit-learn - Build ML models')
print(f'  3. Train/Test Split - Evaluate properly')
print(f'  4. Feature Scaling - Normalize data')
print(f'  5. MLflow - Track experiments')

print(f'\n🎯 Next Steps:')
print(f'  1. Try different algorithms (SVM, XGBoost, Neural Networks)')
print(f'  2. Tune hyperparameters (max_depth, learning_rate)')
print(f'  3. Test on real datasets (Titanic, Credit, House Prices)')
print(f'  4. Deploy model to production (FastAPI, Docker, K8s)')

print(f'\n✨ Congratulations! You\'ve completed a full ML pipeline!')
print('='*60)